# 🚀 TransLSTM-Predictor
**CNN-BiLSTM-Transformer Hybrid Stock Prediction System**

This notebook runs the full pipeline of the TransLSTM-Predictor in a Google Colab GPU environment.

---

## Pipeline
1. Environment Setup & Dependency Installation
2. Data Upload or Sample Data Generation
3. Settings (Hyperparameters)
4. Run Full Pipeline
   - Feature Engineering
   - Walk-forward Validation + Ensemble Training
   - Backtesting
   - Future Prediction

## 1. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Clone repository (first run only)
import os
if not os.path.exists('TransLSTM-Predictor'):
    !git clone https://github.com/irhdab/TransLSTM-Predictor.git

%cd TransLSTM-Predictor
%pip install -U -q -r requirements.txt

## 2. Data Preparation

**Option A**: Upload CSV file directly  
**Option B**: Generate auto sample data in the cell below

In [ ]:
# Option A: File upload
# from google.colab import files
# uploaded = files.upload()  # Requires columns: date, open, high, low, close, volume
# CSV_PATH = list(uploaded.keys())[0]

# Option B: Generate sample data (S&P 500 style simulation)
import numpy as np
import pandas as pd

np.random.seed(42)
days = 1000
dates = pd.bdate_range(start='2020-01-02', periods=days)

# Generate prices based on random walk
returns = np.random.normal(0.0005, 0.015, days)
close = 100 * np.exp(np.cumsum(returns))
high = close * (1 + np.abs(np.random.normal(0, 0.008, days)))
low = close * (1 - np.abs(np.random.normal(0, 0.008, days)))
open_price = close * (1 + np.random.normal(0, 0.003, days))
volume = np.random.randint(1_000_000, 50_000_000, days)

df = pd.DataFrame({
    'date': dates,
    'open': open_price,
    'high': high,
    'low': low,
    'close': close,
    'volume': volume
})

os.makedirs('data', exist_ok=True)
CSV_PATH = 'data/sample_stock.csv'
df.to_csv(CSV_PATH, index=False)
print(f"✓ Sample data generation complete: {df.shape}")
df.tail()

## 3. Settings (Hyperparameters)

In [ ]:
from config import config

# === Modify values here as desired ===
config.EPOCHS = 50            # Reduce in Colab for quick testing
config.ENSEMBLE_SIZE = 3
config.WALK_FORWARD_FOLDS = 3
config.SEQ_LENGTH = 60
config.FUTURE_DAYS = 30
config.RANDOM_SEED = 42

print("=== Current Configuration ===")
print(f"  EPOCHS:             {config.EPOCHS}")
print(f"  ENSEMBLE_SIZE:      {config.ENSEMBLE_SIZE}")
print(f"  WALK_FORWARD_FOLDS: {config.WALK_FORWARD_FOLDS}")
print(f"  SEQ_LENGTH:         {config.SEQ_LENGTH}")
print(f"  FUTURE_DAYS:        {config.FUTURE_DAYS}")
print(f"  PREDICT_RETURNS:    {config.PREDICT_RETURNS}")

## 4. Run Pipeline

In [ ]:
import random
import numpy as np
import tensorflow as tf
from modules.data_loader import DataProcessor
from modules.model_builder import create_lstm_transformer_model as build_model
from modules.trainer import ModelTrainer
from modules.backtester import Backtester

# Fix Seed
random.seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)
tf.random.set_seed(config.RANDOM_SEED)

config.ensure_directories()

# Data Loading
dp = DataProcessor(csv_path=CSV_PATH, config=config)
raw = dp.load_raw_data()
assert dp.validate_data(raw), "Data validation failed!"

data = dp.parse_dates(raw)
data = dp.handle_outliers(data)
original_dates = data['date']
features = dp.extract_features(data)

In [ ]:
# Walk-forward Validation + Ensemble Training
import pandas as pd

seq_length = config.SEQ_LENGTH
future_days = config.FUTURE_DAYS
total_samples = len(features) - seq_length - future_days + 1

all_fold_metrics = []
final_ensemble_models = []

for fold in range(config.WALK_FORWARD_FOLDS):
    print(f"\n{'='*60}")
    print(f"  Fold {fold + 1}/{config.WALK_FORWARD_FOLDS}")
    print(f"{'='*60}")

    train_end_idx = int(total_samples * (0.6 + 0.1 * fold))
    test_end_idx = train_end_idx + (total_samples - train_end_idx) // (config.WALK_FORWARD_FOLDS - fold)

    X_all, y_all, dates_all = dp.create_sequences(features, original_dates)
    norm_features, norm_targets, scaler, target_scaler = dp.normalize_data(features, y_all, train_end=train_end_idx + seq_length)
    sequences_norm, _, _ = dp.create_sequences(norm_features, original_dates)

    fold_X_train = sequences_norm[:train_end_idx]
    fold_y_train = norm_targets[:train_end_idx]
    fold_X_test = sequences_norm[train_end_idx:test_end_idx]
    fold_y_test = norm_targets[train_end_idx:test_end_idx]
    fold_test_dates = dates_all[train_end_idx:test_end_idx].reset_index(drop=True)
    last_prices_indices = np.arange(train_end_idx, test_end_idx) + seq_length - 1
    last_actual_prices = data['close'].iloc[last_prices_indices].values

    fold_models = []
    fold_preds = []

    for m_idx in range(config.ENSEMBLE_SIZE):
        print(f"\n  Training model {m_idx+1}/{config.ENSEMBLE_SIZE}...")
        model = build_model(seq_length=seq_length, num_features=features.shape[1], config=config)
        trainer = ModelTrainer(model, config, scaler, CSV_PATH, target_scaler=target_scaler)
        trainer.compile_model()
        trainer.train(fold_X_train, fold_y_train)
        fold_models.append(model)
        fold_preds.append(model.predict(fold_X_test, verbose=0))

    avg_preds = np.mean(fold_preds, axis=0)

    eval_trainer = ModelTrainer(fold_models[0], config, scaler, CSV_PATH, target_scaler=target_scaler)
    res = eval_trainer.evaluate(
        fold_X_test, fold_y_test, fold_test_dates,
        last_actual_prices=last_actual_prices,
        predictions_override=avg_preds
    )
    fold_mse, fold_mae = res[0], res[1]
    all_fold_metrics.append({'mse': fold_mse, 'mae': fold_mae})

    if fold == config.WALK_FORWARD_FOLDS - 1:
        final_ensemble_models = fold_models
        final_scaler = scaler
        final_target_scaler = target_scaler
        final_test_actual = res[3]
        final_test_pred = res[4]
        final_test_dates = fold_test_dates

# Metrics Summary
metrics_df = pd.DataFrame(all_fold_metrics, index=[f'Fold {i+1}' for i in range(len(all_fold_metrics))])
metrics_df.loc['MEAN'] = metrics_df.mean()
print("\n" + "="*50)
print("  Walk-forward Validation Summary")
print("="*50)
display(metrics_df)

## 5. Backtesting

In [ ]:
backtester = Backtester(config)
bt_results = backtester.run(final_test_actual, final_test_pred, final_test_dates)

## 6. Future Prediction

In [ ]:
# Future Prediction (Ensemble Average)
norm_features_final = final_scaler.transform(features)
last_seq = np.expand_dims(norm_features_final[-seq_length:], axis=0)

future_preds_norm = [m.predict(last_seq, verbose=0) for m in final_ensemble_models]
avg_future = np.mean(future_preds_norm, axis=0)
future_returns = final_target_scaler.inverse_transform(avg_future).flatten()

# Convert returns to prices
last_price = data['close'].iloc[-1]
future_prices = []
p = last_price
for r in future_returns:
    p = p * (1 + r)
    future_prices.append(p)

future_dates = pd.bdate_range(start=original_dates.iloc[-1], periods=config.FUTURE_DAYS + 1)[1:]

# Display
future_df = pd.DataFrame({'date': future_dates, 'predicted_price': future_prices, 'predicted_return': future_returns})
print(f"\n📈 Last actual price: {last_price:.2f}")
display(future_df)

In [ ]:
# Final Plot
final_trainer = ModelTrainer(final_ensemble_models[0], config, final_scaler, CSV_PATH, target_scaler=final_target_scaler)
final_trainer.plot_predictions(
    y_true=final_test_actual,
    y_pred=final_test_pred,
    test_dates=final_test_dates,
    future_predictions=future_prices,
    future_dates=future_dates
)
import matplotlib.pyplot as plt
plt.show()

## 7. Model Save & Download

In [ ]:
import os

# Save models
timestamp = config.get_timestamp()
for i, m in enumerate(final_ensemble_models):
    path = os.path.join(config.MODEL_SAVE_PATH, f'ensemble_{i+1}_{timestamp}.keras')
    m.save(path)
    print(f"✓ Model {i+1} saved: {path}")

# Download (Colab only)
# from google.colab import files
# for f in os.listdir(config.MODEL_SAVE_PATH):
#     if f.endswith('.keras'):
#         files.download(os.path.join(config.MODEL_SAVE_PATH, f))